<a href="https://colab.research.google.com/github/victorlavrenko/answer-engineering/blob/main/notebooks/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quickstart

A minimal demonstration notebook: generate a baseline answer, apply a ruleset
to modify the generation trajectory, and print both baseline and improved
answers to illustrate the effect of rule-based intervention.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DEVICE_MAP = "cuda"  # Run the model on the GPU
DTYPE = "float16"  # Stable and efficient dtype for repeatable inference
MAX_NEW_TOKENS = 1024
VERBOSITY = 1
SYSTEM_PROMPT = (
    "Solve software engineering and architecture tasks by first identifying "
    "the practical design tradeoffs before writing code. "
    "Prefer clear, maintainable designs, but consider simple shortcuts "
    "if they seem reasonable. "
    "Explain your reasoning step by step."
)
QUESTION = (
    "What is the best way to implement rate limiting in a small Python API "
    "web service that may later run on multiple instances?"
)
RULES = """
## Replace (case-sensitive): Implement

With: Consider reusing an existing
"""

In [ ]:
import importlib.util
import os
from pathlib import Path

package = "answer_engineering"
repo = f"{package.replace('_', '-')}"

try:
    from google.colab import userdata  # type: ignore[reportMissingImports]

    t = os.getenv("GITHUB_TOKEN") or userdata.get("GITHUB_TOKEN")  # type: ignore[reportUnknownMemberType]
except Exception:
    t = os.getenv("GITHUB_TOKEN")

if Path(repo).is_dir():
    !git -C {repo} pull
elif Path.cwd().name != "notebooks":
    u = f"https://{t}@github.com/" if t else "https://github.com/"
    !git clone {u}victorlavrenko/{repo}.git
if importlib.util.find_spec(package) is None:
    target = f"{repo}" if Path(repo).is_dir() else ".."
    %pip install -e {target}
    raise SystemExit("Restart runtime required")

In [ ]:
from answer_engineering import GenerationRuntime

runtime = GenerationRuntime(
    MODEL_ID,
    device_map=DEVICE_MAP,
    dtype=DTYPE,
)
runtime.materialize()

In [ ]:
from answer_engineering import (
    GenerationPolicy,
    GenerationRequest,
    GenerationResult,
)

print("=" * 80)
print("BASELINE ANSWER:")
print("=" * 80)
baseline_policy = GenerationPolicy(
    rules="",
    system_prompt=SYSTEM_PROMPT,
    verbosity=VERBOSITY,
    max_new_tokens=MAX_NEW_TOKENS,
)

baseline_request = GenerationRequest(question=QUESTION)
baseline_answer: GenerationResult = runtime.generate(
    baseline_request, baseline_policy
)

print()
print("=" * 80)
print("IMPROVED ANSWER:")
print("=" * 80)
improved_policy = GenerationPolicy(
    rules=RULES,
    system_prompt=SYSTEM_PROMPT,
    verbosity=VERBOSITY,
    max_new_tokens=MAX_NEW_TOKENS,
)

improved_request = GenerationRequest(question=QUESTION)
improved_answer: GenerationResult = runtime.generate(
    improved_request, improved_policy
)

BASELINE ANSWER:
To implement rate limiting in a small Python API web service that may later run
on multiple instances, we need to consider several design trade-offs. Let's
break down the problem and explore the best approach step by step.

### Step 1: Identify the Requirements
1. **Rate Limiting**: Limit the number of requests a client can make within a
certain time frame.
2. **Scalability**: The solution should work well if the service runs on
multiple instances.
3. **Consistency**: Ensure that rate limits are enforced consistently across all
instances.
4. **Performance**: The solution should not significantly impact the performance
of the API.

### Step 2: Evaluate Possible Solutions

#### 1. In-Memory Solutions
- **Simple Dictionary or List**: Store rate limits in a dictionary or list in
memory.
- **Pros**: Easy to implement, fast access.
- **Cons**: Not scalable, each instance would have its own rate limit state,
leading to inconsistent enforcement.

#### 2. Distributed Solutions
